# 05 - Apply eligibility, find players without an IPL team, phase-wise stats
Inputs: outputs/*_ratings.csv (notebook 04), outputs/balls_pressure.parquet (notebook 02), data/player_eligibility_1.csv
Output: outputs/pressure_players_phase_stats.xlsx

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter as L

pd.set_option('display.width', 250, 'display.max_columns', 40, 'display.max_rows', 100)
ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
DATA, OUT = ROOT / 'data', ROOT / 'outputs'

ELIG = 'player_eligibility_5dcc54.csv'     # latest file from the user
E = pd.read_csv(DATA / ELIG).set_index('player_id')

# Safeguard 1: anyone who has batted or bowled for Pakistan / Pakistan A is excluded, whatever the file says.
P0 = pd.read_parquet(OUT / 'balls_pressure.parquet', columns=['striker_id', 'bowler_id', 'bat_team', 'bowl_team'])
PAK = set(P0[P0.bat_team.str.contains('Pakistan')].striker_id) | set(P0[P0.bowl_team.str.contains('Pakistan')].bowler_id)
print('Pakistan-linked ids found in ball data:', len(PAK), '| of which present in eligibility file:', sorted(PAK & set(E.index)))

# Safeguard 2: id 6922 is the PAKISTAN Abdul Samad; the Indian LSG player is id 7446 (row moved to the right id).
if 7446 not in E.index and 6922 in E.index:   # only needed for the older file version
    E.loc[7446] = E.loc[6922]
    E.loc[7446, 'name'] = 'Abdul Samad'

E['status'] = np.select([E.index.isin(PAK), E.current_ipl_team.eq('retired'), E.current_ipl_team.eq('available')],
                        ['excluded (Pakistan)', 'retired', 'no IPL team'], 'has IPL team')
ECOLS = ['team', 'indian_or_overseas', 'capped_indian', 'current_ipl_team', 'status']


def load(path):
    T = pd.read_csv(path, index_col=0)
    T.index.name = 'player_id'
    T = T.join(E[ECOLS], how='left')
    T['status'] = T.status.fillna('excluded (not in eligibility file)')
    return T


BAT, BOWL = load(OUT / 'batter_ratings.csv'), load(OUT / 'bowler_ratings.csv')
print('batters by status :', BAT.status.value_counts().to_dict())
print('bowlers by status :', BOWL.status.value_counts().to_dict())

## Main ranking = players with NO confirmed IPL team ("available"), not retired, not Pakistan
Same consensus score as notebook 04 (so ranks are comparable); pool rank is just the order inside the available pool.

In [ ]:
ACTIVE = ['has IPL team', 'no IPL team']


def pools(T):
    scored = T[T.status.isin(ACTIVE) & T.consensus.notna()].sort_values('consensus', ascending=False).copy()   # >= 50 pressure balls
    scored['sample'] = np.where(scored.eligible, 'meets threshold', 'thin sample')
    act = scored.copy()
    act['rank_active'] = np.arange(1, len(act) + 1)
    av = act[act.status == 'no IPL team'].copy()
    av['rank_available'] = np.arange(1, len(av) + 1)
    pool = T[T.status.eq('no IPL team')]
    thin = pool[pool.consensus.isna() & (pool.raw_p >= 30)].sort_values('pl', ascending=False).copy()   # 30-49 pressure balls
    thin['sample'] = 'watchlist (30-49 balls)'
    ind = pool[pool.indian_or_overseas.eq('indian') & (pool.raw_p >= 30)].sort_values('pl', ascending=False).copy()
    ind['sample'] = np.where(ind.eligible, 'meets threshold', 'thin sample')
    return act, av, thin, ind


bat_act, bat_av, bat_thin, bat_ind = pools(BAT)
bowl_act, bowl_av, bowl_thin, bowl_ind = pools(BOWL)
print('scored available batters:', len(bat_av), '(meeting full threshold: %d)' % bat_av.eligible.sum(), '| bowlers:', len(bowl_av), '(%d)' % bowl_av.eligible.sum())
SHOW = ['name', 'team', 'capped_indian', 'sample', 'raw_p', 'inn_p', 'alpha', 'pl', 'ci_lo', 'ci_hi', 'consensus', 'pressure_type']
print('\nTOP 15 AVAILABLE BATTERS'); print(bat_av.head(15)[['rank_available', 'rank_active'] + SHOW].round(1).to_string())
print('\nTOP 15 AVAILABLE BOWLERS'); print(bowl_av.head(15)[['rank_available', 'rank_active'] + SHOW].round(1).to_string())
print('\nINDIAN AVAILABLE BATTERS (>=30 pressure balls)'); print(bat_ind.head(10)[SHOW].round(1).to_string())
print('\nINDIAN AVAILABLE BOWLERS'); print(bowl_ind.head(10)[SHOW].round(1).to_string())
print('\nTHIN-SAMPLE WATCHLIST - batters'); print(bat_thin.head(6)[SHOW].round(1).to_string())
print('\nTHIN-SAMPLE WATCHLIST - bowlers'); print(bowl_thin.head(6)[SHOW].round(1).to_string())

## Phase-wise counts from ball-by-ball (powerplay 1-6, middle 7-15, death 16-20; regulation innings only)
Batting: runs off the bat, balls faced (wides excluded, no-balls included), dismissals incl. run-outs, retired-out excluded.
Bowling: legal balls, runs conceded (bat runs + wides + no-balls), wickets credited to the bowler (no run-outs).

In [ ]:
P = pd.read_parquet(OUT / 'balls_pressure.parquet', columns=[
    'innings', 'phase', 'striker_id', 'bowler_id', 'runs', 'faced', 'legal', 'bowler_runs', 'bowler_wkt',
    'dismissed_player_id', 'dismissal_type', 'is_wicket', 'match_tier'])
P = P[P.innings.isin([1, 2])]
PH = ['PP', 'MID', 'DEATH']

bt = P[P.faced].groupby(['striker_id', 'phase']).agg(balls=('runs', 'size'), runs=('runs', 'sum'))
outs = P[P.is_wicket & P.dismissal_type.ne('Retire Out')].groupby(['dismissed_player_id', 'phase']).size().rename('outs')
outs.index.names = ['striker_id', 'phase']
BATP = bt.join(outs, how='outer').fillna(0).unstack('phase').reindex(columns=PH, level=1).fillna(0)

bw = P.groupby(['bowler_id', 'phase']).agg(balls=('legal', 'sum'), runs=('bowler_runs', 'sum'), wkts=('bowler_wkt', 'sum'))
BOWLP = bw.unstack('phase').reindex(columns=PH, level=1).fillna(0)

weak = P.match_tier.isin(['T20I_ASSOC', 'A_TEAM'])
weak_bat = P[P.faced & weak].groupby('striker_id').size() / P[P.faced].groupby('striker_id').size()
weak_bowl = P[weak].groupby('bowler_id').size() / P.groupby('bowler_id').size()

## Excel export (counts are values; SR / average / economy are live formulas)

In [ ]:
F = Font(name='Arial', size=10)
HF = Font(name='Arial', size=10, bold=True, color='FFFFFF')
HFILL = PatternFill('solid', start_color='1F3864')
BFILL = PatternFill('solid', start_color='D9E1F2')
wb = Workbook()


def write_table(ws, df, cols, headers, start_row=1, number_formats=None):
    for j, h in enumerate(headers, 1):
        c = ws.cell(row=start_row, column=j, value=h)
        c.font, c.fill, c.alignment = HF, HFILL, Alignment(horizontal='center', vertical='center', wrap_text=True)
    for i, (_, r) in enumerate(df[cols].iterrows(), start_row + 1):
        for j, col in enumerate(cols, 1):
            v = r[col]
            v = None if (isinstance(v, float) and np.isnan(v)) else (v.item() if hasattr(v, 'item') else v)
            c = ws.cell(row=i, column=j, value=v)
            c.font = F
            if number_formats and col in number_formats and v is not None:
                c.number_format = number_formats[col]
    ws.freeze_panes = ws.cell(row=start_row + 1, column=3)
    for j in range(1, len(cols) + 1):
        ws.column_dimensions[L(j)].width = 14 if j > 2 else (8 if j == 1 else 24)
    ws.row_dimensions[start_row].height = 32


NF = {'raw_p': '0', 'inn_p': '0', 'alpha': '0.0', 'pl': '0.0', 'ci_lo': '0.0', 'ci_hi': '0.0', 'consensus': '0.00', 'weak_share': '0%'}
RC = ['name', 'team', 'indian_or_overseas', 'capped_indian', 'current_ipl_team', 'raw_p', 'inn_p', 'alpha', 'pl', 'ci_lo', 'ci_hi', 'consensus', 'pressure_type', 'weak_share']
RH = ['Player', 'National team', 'Indian / overseas', 'Capped (Indian)', 'IPL team', 'Pressure balls', 'Pressure innings', 'Baseline skill', 'Pressure rating', 'CI low', 'CI high',
      'Consensus score', 'Pressure type (indicative)', 'Share of pressure balls vs weak tiers']


def rank_sheet(name, df, rank_col, n):
    ws = wb.create_sheet(name)
    d = df.head(n).reset_index()
    d['name_'] = d['name']
    write_table(ws, d, [rank_col, 'name', 'sample'] + RC[1:], ['Rank', 'Player', 'Sample'] + RH[1:], number_formats=NF)
    return d


ws0 = wb.active
ws0.title = 'README'
notes = [
    'Pressure players - AVAILABLE pool (no confirmed IPL team): rankings and phase-wise stats',
    'Source: notebooks 01-05 (ball_by_ball.csv, matches.csv, player_eligibility_1.csv supplied by the user).',
    'Active pool = player is in player_eligibility_1.csv, not marked retired, and has never played for Pakistan / Pakistan A in the ball data (8 such players were found in the file and excluded; the Indian Abdul Samad = id 7446).',
    'No IPL team = current_ipl_team is "available" in the eligibility file, i.e. no confirmed team as of now. It does NOT mean registered for the auction. Retention / release data is not applied yet (user decision).',
    'Pressure = top 20% of leverage within competition tier (innings 2: win-probability swing; innings 1: projected score vs par). Super overs count as maximum pressure.',
    'Pressure rating, baseline skill and CI are runs-equivalent per 100 balls vs an average player in the same match state (empirical-Bayes shrunk).',
    'Main sheets rank only players with no confirmed IPL team. Reference sheets show the same ranking over all active players. Rank = consensus score across 12 specifications (pressure threshold 70/80/90, weak-tier weight 0.5/1.0, consistency weight 0.1/0).',
    'Scored = >=50 pressure balls. Sample = meets threshold if >=60 pressure balls and >=8 pressure innings, otherwise thin sample (rating is shrunk toward the player baseline; check the CI). Watchlist = no IPL team, >=30 pressure balls but below that threshold (wide uncertainty). Indian sheet lists every Indian with no IPL team and >=30 pressure balls, ranked by pressure rating.',
    'Phase stats: regulation innings of IPL and T20I data from 2022 on, incl. associate and A-team games (see "vs weak tiers" share). PP = overs 1-6, MID = 7-15, DEATH = 16-20.',
    'Batting: SR = runs / balls faced x 100; Avg = runs / dismissals (run-outs included, retired-out excluded); "-" = not out in that phase.',
    'Bowling: Econ = runs per over; Avg = runs per wicket; SR = balls per wicket (run-outs not credited to the bowler); "-" = no wickets.',
    'Pressure type (Thrives / Dips) is indicative only: the pressure-vs-baseline gap had low split-half reliability.',
]
for i, t in enumerate(notes, 1):
    c = ws0.cell(row=i, column=1, value=t)
    c.font = Font(name='Arial', size=12 if i == 1 else 10, bold=(i == 1))
ws0.column_dimensions['A'].width = 160

a = rank_sheet('Available - batters', bat_av, 'rank_available', 25)
b_ = rank_sheet('Available - bowlers', bowl_av, 'rank_available', 25)
wi = wb.create_sheet('Indian available')
ii = pd.concat([bat_ind.head(12).assign(role='Batter'), bowl_ind.head(12).assign(role='Bowler')]).reset_index()
write_table(wi, ii, ['role', 'name', 'sample'] + RC[1:], ['Role', 'Player', 'Sample'] + RH[1:], number_formats=NF)
ww = wb.create_sheet('Watchlist (thin sample)')
wl = pd.concat([bat_thin.head(10).assign(role='Batter'), bowl_thin.head(10).assign(role='Bowler')]).reset_index()
write_table(ww, wl, ['role', 'name', 'sample'] + RC[1:], ['Role', 'Player', 'Sample'] + RH[1:], number_formats=NF)
c = rank_sheet('Ref - top batters (all active)', bat_act, 'rank_active', 20)
d = rank_sheet('Ref - top bowlers (all active)', bowl_act, 'rank_active', 20)


def phase_sheet(title, ids, roles):
    ws = wb.create_sheet(title)
    bat = roles == 'bat'
    base = ['Player', 'National team', 'IPL team']
    metrics = ['Balls', 'Runs', 'Outs', 'SR', 'Avg'] if bat else ['Balls', 'Runs', 'Wkts', 'Econ', 'Avg', 'SR']
    blocks = ['Overall'] + ['Powerplay (1-6)', 'Middle (7-15)', 'Death (16-20)']
    ncols = len(metrics)
    for j, h in enumerate(base, 1):
        ws.cell(row=2, column=j, value=h)
    for k, blk in enumerate(blocks):
        c0 = 4 + k * ncols
        ws.cell(row=1, column=c0, value=blk)
        ws.merge_cells(start_row=1, start_column=c0, end_row=1, end_column=c0 + ncols - 1)
        for m, h in enumerate(metrics):
            ws.cell(row=2, column=c0 + m, value=h)
    last = 4 + len(blocks) * ncols
    ws.cell(row=2, column=last, value='% balls vs weak tiers')
    for row in ws.iter_rows(min_row=1, max_row=2, max_col=last):
        for cell in row:
            cell.font, cell.fill, cell.alignment = HF, HFILL, Alignment(horizontal='center', vertical='center', wrap_text=True)
    T = BAT if bat else BOWL
    DATAP = BATP if bat else BOWLP
    for i, pid in enumerate(ids, 3):
        ws.cell(row=i, column=1, value=T.loc[pid, 'name'])
        ws.cell(row=i, column=2, value=T.loc[pid, 'team'])
        ws.cell(row=i, column=3, value=T.loc[pid, 'current_ipl_team'])
        cnt = DATAP.loc[pid] if pid in DATAP.index else None
        for k in range(len(blocks)):
            c0 = 4 + k * ncols
            if bat:
                vals = [(cnt['balls', ph] if cnt is not None else 0) for ph in PH], [(cnt['runs', ph] if cnt is not None else 0) for ph in PH], [(cnt['outs', ph] if cnt is not None else 0) for ph in PH]
            else:
                vals = [(cnt['balls', ph] if cnt is not None else 0) for ph in PH], [(cnt['runs', ph] if cnt is not None else 0) for ph in PH], [(cnt['wkts', ph] if cnt is not None else 0) for ph in PH]
            if k == 0:   # overall = sum of the three phase blocks (live formulas)
                for m in range(3):
                    refs = '+'.join('%s%d' % (L(4 + (p + 1) * ncols + m), i) for p in range(3))
                    ws.cell(row=i, column=c0 + m, value='=' + refs)
            else:
                for m in range(3):
                    ws.cell(row=i, column=c0 + m, value=float(vals[m][k - 1]))
            B, R_, W = (L(c0 + x) + str(i) for x in range(3))
            if bat:
                ws.cell(row=i, column=c0 + 3, value='=IF(%s=0,"",%s/%s*100)' % (B, R_, B))
                ws.cell(row=i, column=c0 + 4, value='=IF(%s=0,"-",%s/%s)' % (W, R_, W))
            else:
                ws.cell(row=i, column=c0 + 3, value='=IF(%s=0,"",%s/%s*6)' % (B, R_, B))
                ws.cell(row=i, column=c0 + 4, value='=IF(%s=0,"-",%s/%s)' % (W, R_, W))
                ws.cell(row=i, column=c0 + 5, value='=IF(%s=0,"-",%s/%s)' % (W, B, W))
        wk = (weak_bat if bat else weak_bowl).get(pid, 0)
        ws.cell(row=i, column=last, value=float(wk) if wk == wk else 0.0)
        for j in range(1, last + 1):
            cell = ws.cell(row=i, column=j)
            cell.font = F
            if j > 3 and j < last:
                cell.number_format = '0.0' if ((j - 4) % ncols) >= 3 else '0'
            if j == last:
                cell.number_format = '0%'
    ws.freeze_panes = 'D3'
    ws.column_dimensions['A'].width = 24
    ws.column_dimensions['B'].width = 14
    ws.column_dimensions['C'].width = 26
    for j in range(4, last + 1):
        ws.column_dimensions[L(j)].width = 8
    ws.column_dimensions[L(last)].width = 12
    ws.row_dimensions[2].height = 32
    return ws


bat_ids = list(dict.fromkeys(list(bat_av.head(25).index) + list(bat_ind.head(12).index) + list(bat_thin.head(10).index)))
bowl_ids = list(dict.fromkeys(list(bowl_av.head(25).index) + list(bowl_ind.head(12).index) + list(bowl_thin.head(10).index)))
phase_sheet('Phase stats - batters', bat_ids, 'bat')
phase_sheet('Phase stats - bowlers', bowl_ids, 'bowl')
wb.save(OUT / 'pressure_players_phase_stats.xlsx')
print('\nsaved', OUT / 'pressure_players_phase_stats.xlsx', '| batters in phase sheet:', len(bat_ids), '| bowlers:', len(bowl_ids))